# Custom Synthetic Test Data Generation for RAG Evaluation

> 本 Notebook 演示：**从原始文档自动生成 RAG 评测集**，并用 Ragas 指标对比两种生成方式的效果。

## 整体流程

```
原始文档 → 切分 Chunk → 合成 QA 测试集 → 搭建 RAG → Ragas 评估
                              ├─ Ragas TestsetGenerator（官方）
                              └─ 自定义 LLM 逐 Chunk 生成（可控）
```

## 前置要求

- Python 3.10+
- `.env` 中配置 `OPENAI_API_KEY`、`GOOGLE_API_KEY`（Embedding）
- 主要依赖：`ragas`、`langchain`、`datasets`、`faiss-cpu`

---

## Part 1 — 环境配置

初始化 LLM、Embedding 模型，并包装为 Ragas 可用的接口。Embedding 使用**本地文件缓存**，避免重复调用 API。

In [2]:
!python --version

Python 3.11.0


In [5]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [9]:
embeddings

GoogleGenerativeAIEmbeddings(client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000002B2BA262350>, async_client=None, model='models/gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, client_options=None, base_url=None, transport=None, request_options=None)

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings

store = LocalFileStore("embedding_cache/")
if  isinstance(embeddings, GoogleGenerativeAIEmbeddings):
    name_space = embeddings.model
else:
    name_space = embeddings.model_name

cached_embedding = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embeddings,
    document_embedding_cache=store,
    namespace=name_space
)

d:\Programs\.pyenv\pyenv-win\versions\3.11.0\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [10]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(llm)
generator_embeddings = LangchainEmbeddingsWrapper(cached_embedding)

---

## Part 2 — 加载语料

从 HuggingFace 加载新闻分类数据集 `valurank/News_Articles_Categorization`，取前 **10 条**作为演示语料。

每条样本包含 `Text`（正文）和 `Category`（类别）。先查看文本长度，了解后续切分规模。

In [11]:
from datasets import load_dataset

ds = load_dataset("valurank/News_Articles_Categorization")
my_ds = ds["train"].select(range(10))
my_ds

Dataset({
    features: ['Text', 'Category'],
    num_rows: 10
})

In [15]:
for d in my_ds['Text']:
    print(len(d))
    print(d[:100])

721
Elon Musk, Amber Heard Something's Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Hear
4749
Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of wh
721
Jared Fogle Shut Down By Judge In Bid for Early Release 1/22/2018 Jared Fogle got no love from the j
5565
The agency had come under fire from members of Congress and other groups for allowing dozens of wild
13988
Credit...Jim Wilson/The New York TimesJune 30, 2018WASHINGTON On the final day of the Supreme Court 
14264
Before Coming Out, a Hard Time Growing UpVideoMichael Sam, a defensive end at Missouri who will ente
7950
Credit...Laurent Cipriani/Associated PressNov. 20, 2018BRUSSELS Gathered at a glitzy Dubai resort th
5138
Fortnites parent company, Epic Games, had broken its contract with Apple, a federal judge found. The
771
Lil Pump Attention Record Companies ... $15 Mil or Take a Hike 1/27/2018 TMZ.com Lil Pump is super h
10305
The InterpreterCredit...Jean-Paul Pelissi

---

## Part 3 — 文档切分（Chunking）

使用 `RecursiveCharacterTextSplitter` 将长文切分为适合检索的片段：

| 参数 | 值 | 说明 |
|------|-----|------|
| `chunk_size` | 350 tokens | 按 tiktoken 计数，非字符数 |
| `chunk_overlap` | 70 tokens | 相邻 chunk 重叠，减少边界截断 |
| `separators` | 段落 → 句号 | 优先在语义边界切分 |

每个 chunk 封装为 LangChain `Document`，metadata 记录 `doc_index` / `chunk_index`，便于追溯来源。

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain_core.documents import Document

tokenizer = tiktoken.get_encoding("cl100k_base")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=70,
    separators=["\n\n", "\n", ".", "!", "?"],
    length_function=lambda x: len(tokenizer.encode(x))
)

all_chunks2 = []
for i, t in enumerate(my_ds['Text']):
    chunks = text_splitter.split_text(t)
    total_chunks = len(chunks)
    for j, chunk_text in enumerate(chunks):
        all_chunks2.append(Document(
            page_content=chunk_text,
            metadata={"source":i, "doc_index": i, "chunk_index": j, "total_chunks": total_chunks}
        ))
        print(chunk_text)
        print("="*60)
print(f"Total chunks: {len(all_chunks2)}")


Elon Musk, Amber Heard Something's Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we're not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that's definitely on again. But that's only because that's exactly what they are -- no matter how many times they try to say they're not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they've smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com
Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of which are well-established and some of which have never been approved for medical use before. Most of these vaccines target the so-called spike proteins that cover the virus and he

In [ ]:
for c in all_chunks2:
    print(c.metadata['doc_index'], c.metadata['chunk_index'], c.page_content)
    print('\n========\n')

0 0 Elon Musk, Amber Heard Something's Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we're not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that's definitely on again. But that's only because that's exactly what they are -- no matter how many times they try to say they're not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they've smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com


1 0 Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of which are well-established and some of which have never been approved for medical use before. Most of these vaccines target the so-called spike proteins that cover the vi

---

## Part 4 — Ragas 官方测试集生成

使用 Ragas `TestsetGenerator` 从 chunk 列表**自动合成**评测样本。

**内部流程（简化）：**

1. 对 chunk 做摘要、过滤、主题/实体抽取
2. 构建 chunk 间相似度图
3. 生成 Persona → 生成查询场景 → 合成 QA 对

本例仅启用 `SingleHopSpecificQuerySynthesizer`（单跳、具体问题），生成 5 条样本。每条包含：

- `user_input` — 用户问题
- `reference` — 标准答案
- `reference_contexts` — 出题依据的原文片段

In [18]:
from ragas.testset import TestsetGenerator
from ragas.testset.persona import Persona
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(), 1),
    # (MultiHopSpecificQuerySynthesizer(), 0.3),
    # (MultiHopAbstractQuerySynthesizer(), 0.1),
]

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)


testset = generator.generate_with_langchain_docs(
    all_chunks2,
    testset_size=5,
    query_distribution=query_distribution,
)
testset

Applying SummaryExtractor:   0%|          | 0/49 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/49 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/147 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/5 [00:00<?, ?it/s]

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='What recent activities have Elon Musk and Amber Heard engaged in while in WeHo?', retrieved_contexts=None, reference_contexts=['Elon Musk, Amber Heard Something\'s Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we\'re not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that\'s definitely on again. But that\'s only because that\'s exactly what they are -- no matter how many times they try to say they\'re not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they\'ve smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com'], response=None, multi_responses=None, reference='Elon Musk and Amber 

### 4B · 查看生成的测试集

转为 DataFrame 查看 `user_input`、`reference`、`reference_contexts` 和 `synthesizer_name`。

In [19]:
df = testset.to_pandas()
df

,user_input,reference_contexts,reference,synthesizer_name
0,What recent activities have Elon Musk and Ambe...,"[Elon Musk, Amber Heard Something's Fishy On W...",Elon Musk and Amber Heard were spotted on a su...,single_hop_specifc_query_synthesizer
1,How are scientists approaching the development...,[Scientists are developing more than 100 coron...,Scientists are developing more than 100 corona...,single_hop_specifc_query_synthesizer
2,How did DNA vaccines protect monkeys from the ...,[. DNA Vaccines A number of experimental coron...,Prototype DNA vaccines based on the spike prot...,single_hop_specifc_query_synthesizer
3,What role does Moderna play in the development...,"[. COMPANIES: Moderna, Pfizer and BioNTech, Cu...",Moderna is one of the companies involved in th...,single_hop_specifc_query_synthesizer
4,How does the National Institute of Allergy and...,"[. COMPANIES: Medicago, Doherty Institute and ...",The National Institute of Allergy and Infectio...,single_hop_specifc_query_synthesizer


In [20]:
for index in range(len(df)):
    print("index:", index)
    print("user_input:", df.iloc[index]['user_input'])
    print("reference_contexts:", df.iloc[index]['reference_contexts'])
    print("reference:", df.iloc[index]['reference'])
    print("synthesizer_name:", df.iloc[index]['synthesizer_name'])
    print("="*60)

index: 0
user_input: What recent activities have Elon Musk and Amber Heard engaged in while in WeHo?
reference_contexts: ['Elon Musk, Amber Heard Something\'s Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we\'re not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that\'s definitely on again. But that\'s only because that\'s exactly what they are -- no matter how many times they try to say they\'re not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they\'ve smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com']
reference: Elon Musk and Amber Heard were spotted on a sushi date in WeHo, where they appeared to be a full-blown hand-holding couple, suggesting 

---

## Part 5 — 构建 RAG 流水线

将 chunk 写入 **FAISS** 向量库，构建检索 + 生成链路：

```
用户问题 → FAISS 检索 (top-5) → LLM 生成答案
```

后续可选叠加 **ContextualCompressionRetriever**（过滤 + 抽取 + Rerank），本 Notebook 基线先用纯向量检索。

In [22]:
from langchain_community.vectorstores import FAISS


vectorstore = FAISS.from_documents(
    documents = all_chunks2,
    embedding = cached_embedding
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

### 5B · 进阶：压缩 + Rerank 检索器（可选）

`ContextualCompressionRetriever` 在基础检索之上串联：

1. **LLMChainFilter** — 过滤不相关 chunk
2. **LLMChainExtractor** — 抽取与问题相关的核心句
3. **LLMListwiseRerank** — 对剩余文档重新排序

> 下方 cell 定义了该 retriever，基线评估默认使用上方简单 `retriever`。

In [ ]:
from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline,
    LLMChainExtractor,
    LLMChainFilter,
    LLMListwiseRerank
)
from langchain_classic.retrievers import ContextualCompressionRetriever

# 过滤掉不相关或无用的文本 chunk
llm_filter = LLMChainFilter.from_llm(llm)
# 从 chunk 中抽取核心信息
llm_extractor = LLMChainExtractor.from_llm(llm)
# 文档压缩流水线
compressor_pipeline = DocumentCompressorPipeline(
    transformers=[llm_filter, llm_extractor]
)
# Reranker
reranker = LLMListwiseRerank.from_llm(llm)
compression_rerank_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor_pipeline,
    reranker=reranker
)

### 5C · RetrievalQA 链

使用严格约束的 Prompt：**仅依据 context 回答**，context 不足时回答 `"I don't know"`。这有助于在评估中区分「检索失败」与「生成幻觉」。

In [25]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


prompt_template = """
Answer strictly based on the context below.
Do NOT use information outside the context. 
If context does not contain the answer, say "I don't know".
Keep answer < 2 sentences.

Context:
{context}

Question:
{question}
"""

def get_qa_chain(retriever):
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",  
        retriever= retriever,
        return_source_documents=True, 
        chain_type_kwargs={"prompt": PromptTemplate(input_variables=["context","question"], template=prompt_template)}
    )
chain = get_qa_chain(retriever)

---

## Part 6 — 基线评估（Ragas 官方测试集）

对 Part 4 生成的测试集逐条调用 RAG，再用 Ragas 计算四项核心指标：

| 指标 | 衡量什么 |
|------|----------|
| **Context Precision** | 检索到的 context 是否相关 |
| **Context Recall** | 是否召回了回答所需的关键信息 |
| **Faithfulness** | 生成答案是否忠于 context（无幻觉） |
| **Answer Relevancy** | 答案与问题的相关程度 |

> 观察：当 reference 仅提及机构名称（如 NIAID）而 context 只有 "Sources: ..." 列表时，RAG 可能正确回答 "I don't know"，但 **Context Recall** 会偏低——这是评测集质量与检索粒度的典型张力。

In [29]:
responses_1 = [chain.invoke(x.eval_sample.user_input) for x in testset.samples]
responses_1

[{'query': 'What recent activities have Elon Musk and Amber Heard engaged in while in WeHo?',
  'result': 'They went on a sushi date and looked like a couple, holding hands and being close.',
  'source_documents': [Document(id='3e0aef2c-fe8e-4652-9bb8-033efd090ac8', metadata={'source': 0, 'doc_index': 0, 'chunk_index': 0, 'total_chunks': 1}, page_content='Elon Musk, Amber Heard Something\'s Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we\'re not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that\'s definitely on again. But that\'s only because that\'s exactly what they are -- no matter how many times they try to say they\'re not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they\'ve smooched and gone danc

In [32]:
for r in responses_1:
    print(r['result'])
    print('='*60)

They went on a sushi date and looked like a couple, holding hands and being close.
Scientists are developing more than 100 coronavirus vaccines using a range of techniques, including well-established methods like inactivated vaccines and innovative approaches such as genetic, viral vector, protein-based, and recombinant vaccines.
Prototype DNA vaccines based on the spike protein protected monkeys from the coronavirus by stimulating the immune system to produce antibodies against the virus.
Moderna is involved in developing RNA vaccines for COVID-19, delivering messenger RNA into cells to provoke an immune response.
I don't know.


### 6B · 组装 EvaluationDataset 并调用 `evaluate()`

In [33]:
from ragas.metrics import answer_relevancy, faithfulness, context_recall, ContextPrecision

metrics=[ContextPrecision(), context_recall, faithfulness, answer_relevancy]

In [34]:
from ragas import EvaluationDataset
from ragas import evaluate

def gen_evaluate_dataset(testset, responses):
    hfset = testset.to_hf_dataset()
    records = []
    for sample, resp in zip(hfset, responses):
        records.append({
            "user_input": sample['user_input'],
            "reference": sample['reference'],
            "response": resp['result'],
            "retrieved_contexts": [d.page_content for d in resp['source_documents']],
        })
    eval_dataset = EvaluationDataset.from_dict(records)
    return eval_dataset

In [38]:
eval_dataset = gen_evaluate_dataset(testset, responses_1)

In [39]:
eval_results  = evaluate(
    dataset=eval_dataset, 
    metrics=metrics,
    llm=llm,
    embeddings=cached_embedding
)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

In [42]:
eval_results

{'context_precision': 0.8500, 'context_recall': 0.8000, 'faithfulness': 0.7000, 'answer_relevancy': 0.6041}

In [45]:
eval_results['answer_relevancy']

[np.float64(0.6516946369591511),
 np.float64(0.8093610615271499),
 np.float64(0.7815123294287667),
 np.float64(0.7777963744149187),
 np.float64(0.0)]

In [46]:
responses_1[-1]

{'query': 'How does the National Institute of Allergy and Infectious Diseases contribute to the development of COVID-19 vaccines?',
 'result': "I don't know.",
 'source_documents': [Document(id='8944d89e-92a0-418b-bee6-8e6a6ef88436', metadata={'source': 1, 'doc_index': 1, 'chunk_index': 0, 'total_chunks': 4}, page_content='Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of which are well-established and some of which have never been approved for medical use before. Most of these vaccines target the so-called spike proteins that cover the virus and help it invade human cells. The immune system can develop antibodies that latch onto spike proteins and stop the virus. A successful vaccine for the SARS-CoV-2 coronavirus would teach peoples immune systems to make antibodies against the virus without causing disease. Whole-Virus Vaccines Vaccines that modify the entire coronavirus to provoke an immune response. Inactivated and Live Attenuated Va

---

## Part 7 — 自定义 QA 生成（可控方案）

相比 Ragas 全自动流程，这里用 **LLM 逐 chunk 生成 QA**，优势：

- 逻辑透明、易于调试
- 可控制每文档 / 每 chunk 的 QA 数量
- `reference_contexts` 精确绑定到单个 chunk（评测更干净）

**策略：**

- 每个文档最多 `max_qa_per_doc=1` 条
- 每个 chunk 最多 `max_qa_per_chunk=1` 条
- LLM 输出 JSON 数组 `[{"query": "...", "answer": "..."}]`

生成后包装为 Ragas `Testset`，与 Part 4 格式对齐，便于同口径对比。

In [53]:
import json

def generate_qa_from_chunks(llm, chunks, target_qa_count = 10, max_qa_per_doc=1, max_qa_per_chunk=1) -> list[dict]:
    all_qa = []
    doc_qa_counter = {}
    for chunk in chunks:
        source = chunk.metadata['source']
        doc_qa_count = doc_qa_counter.get(source, 0)
        remaining_for_doc = max_qa_per_doc - doc_qa_counter.get(source, 0)
        if remaining_for_doc <= 0:
            continue
        max_qa=min(max_qa_per_chunk, remaining_for_doc)
        qas = generate_qa_from_chunk(llm, chunk, max_qa_per_chunk=max_qa)
        print("Generated QA count:", len(qas), chunk.metadata['source'], chunk.metadata['doc_index'], chunk.metadata['chunk_index'])
        all_qa.extend(qas)
        doc_qa_counter[source] = doc_qa_count + len(qas)
        if len(all_qa) >= target_qa_count:
            break
    return all_qa[:target_qa_count]


def generate_qa_from_chunk(llm, doc_chunk, max_qa_per_chunk: int = 1) -> list[dict]:
    prompt = f"""
You are an assistant. Analyze the text below and generate up to {max_qa_per_chunk} question(s)
with their answers based strictly on the text.

If the text is unsuitable for generating QA, return an empty JSON array.

Always output ONLY in JSON array format, like:
[
    {{"query": "question1", "answer": "answer1"}},
    ...
]

Text:
{doc_chunk.page_content}
"""
    response = llm.invoke(prompt).content
    try:
        raw_qas = json.loads(response)
    except Exception as e:
        print("Failed to parse LLM output:", e)
        print("LLM output:", response)
        return []
    
    results = []
    for qa in raw_qas:
        if "query" not in qa or "answer" not in qa:
            continue
        results.append({
            "user_input": qa["query"],
            "reference": qa["answer"],
            "retrieved_contexts": [doc_chunk.page_content],
            "doc_index": doc_chunk.metadata['doc_index'],
            "chunk_index": doc_chunk.metadata['chunk_index'],
            "synthesizer_name": "single_hop_specifc_query_synthesizer"

        })
    return results

In [54]:
from ragas.testset.synthesizers.testset_schema import Testset

testset_l = generate_qa_from_chunks(llm, all_chunks2, target_qa_count=5)
testset3 = Testset.from_list(testset_l)

Generated QA count: 1 0 0 0
Generated QA count: 1 1 1 0
Generated QA count: 1 2 2 0
Generated QA count: 1 3 3 0
Generated QA count: 1 4 4 0


---

## Part 8 — 对比评估：自定义测试集

用同一套 RAG 流水线评估 Part 7 生成的测试集，与 Part 6 结果对比。

**预期差异：**

- 自定义方案的 `reference_contexts` 通常只有 **1 个 chunk**，Context Recall / Faithfulness 往往更高
- Ragas 自动生成的 question 可能更「绕」，对检索要求更高

对比两次 `eval_results` 的各指标均值，选择更适合你业务的测试集生成策略。

In [55]:
responses_3 = [chain.invoke(x.eval_sample.user_input) for x in testset3.samples]
responses_3

[{'query': 'Are Elon Musk and Amber Heard currently back together according to the text?',
  'result': 'Yes, according to the text, Elon Musk and Amber Heard are effectively back together, as they are described as "the full-blown hand-holding couple that\'s definitely on again" despite claims they are not reunited.',
  'source_documents': [Document(id='3e0aef2c-fe8e-4652-9bb8-033efd090ac8', metadata={'source': 0, 'doc_index': 0, 'chunk_index': 0, 'total_chunks': 1}, page_content='Elon Musk, Amber Heard Something\'s Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we\'re not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that\'s definitely on again. But that\'s only because that\'s exactly what they are -- no matter how many times they try to say they\'re not reunited. We broke the story ... Elon and Amber start

In [58]:
eval_dataset = gen_evaluate_dataset(testset3, responses_3)
eval_results3  = evaluate(
    dataset=eval_dataset, 
    metrics=metrics,
    llm=llm,
    embeddings=cached_embedding
)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

In [66]:
testset3.to_pandas()

,user_input,retrieved_contexts,reference,synthesizer_name
0,Are Elon Musk and Amber Heard currently back t...,"[Elon Musk, Amber Heard Something's Fishy On W...","Yes, the text suggests that Elon Musk and Ambe...",single_hop_specifc_query_synthesizer
1,What do most coronavirus vaccines target to he...,[Scientists are developing more than 100 coron...,Most vaccines target the spike proteins that c...,single_hop_specifc_query_synthesizer
2,Why did Judge Tanya Pratt reject Jared Fogle's...,[Jared Fogle Shut Down By Judge In Bid for Ear...,Judge Tanya Pratt rejected Jared Fogle's legal...,single_hop_specifc_query_synthesizer
3,What action did the FDA announce regarding cor...,[The agency had come under fire from members o...,The FDA announced that companies selling coron...,single_hop_specifc_query_synthesizer
4,How have conservative groups used the First Am...,[Credit...Jim Wilson/The New York TimesJune 30...,Conservative groups have used the First Amendm...,single_hop_specifc_query_synthesizer


In [72]:
testset3.samples[2].eval_sample.retrieved_contexts

['Jared Fogle Shut Down By Judge In Bid for Early Release 1/22/2018 Jared Fogle got no love from the judge who sentenced him to 15 years in prison ... ruling his legal arguments were BS. Fogle had filed legal docs he prepared himself in the prison library, claiming Judge Tanya Pratt had no jurisdiction to even hear the criminal case because the sex crime charges required proof he traveled from one state to another to engage in sex acts with minors ... and he says everything was done in 1 state. Judge Pratt begged to differ, repeatedly calling Fogle\'s argument "frivolous," saying federal law specifically covers his crimes. The judge invited him to appeal to a higher court, but made it clear ... that dog don\'t hunt.']

In [59]:
eval_results3

{'context_precision': 0.9400, 'context_recall': 1.0000, 'faithfulness': 0.9333, 'answer_relevancy': 0.8622}

In [64]:
eval_results3['faithfulness']

[1.0, 1.0, 0.6666666666666666, 1.0, 1.0]

In [65]:
responses_3[2]

{'query': "Why did Judge Tanya Pratt reject Jared Fogle's legal arguments for early release?",
 'result': 'Judge Tanya Pratt rejected Jared Fogle\'s legal arguments for early release because she found his claims "frivolous" and stated that federal law specifically covers his crimes.',
 'source_documents': [Document(id='09d97abb-1a89-4d25-86ce-5b333bed59df', metadata={'source': 2, 'doc_index': 2, 'chunk_index': 0, 'total_chunks': 1}, page_content='Jared Fogle Shut Down By Judge In Bid for Early Release 1/22/2018 Jared Fogle got no love from the judge who sentenced him to 15 years in prison ... ruling his legal arguments were BS. Fogle had filed legal docs he prepared himself in the prison library, claiming Judge Tanya Pratt had no jurisdiction to even hear the criminal case because the sex crime charges required proof he traveled from one state to another to engage in sex acts with minors ... and he says everything was done in 1 state. Judge Pratt begged to differ, repeatedly calling Fo